# 04. Aggregations, Joins, and Window Functions

This notebook covers the patterns you use most often in analytics and ETL pipelines.

## Topics
- `groupBy` and `agg`
- join types
- ranking and window functions
- cumulative metrics and running totals


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('AggregationsAndJoins').master('local[*]').getOrCreate()

sales = spark.createDataFrame([
    ('A', 'Laptop', 1200),
    ('A', 'Phone', 700),
    ('B', 'Laptop', 1500),
    ('B', 'Tablet', 600),
    ('C', 'Phone', 800)
], ['region', 'product', 'amount'])

sales.groupBy('region').agg(F.sum('amount').alias('total_sales')).show()
spark.stop()


## Joins

Joins are essential when combining fact and dimension tables. Common types include inner, left, right, outer, semi, and anti.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('JoinDemo').master('local[*]').getOrCreate()

orders = spark.createDataFrame([
    (1, 101, 2),
    (2, 102, 1),
    (3, 103, 3)
], ['order_id', 'customer_id', 'qty'])

customers = spark.createDataFrame([
    (101, 'Alice'),
    (102, 'Bob'),
    (104, 'Dana')
], ['customer_id', 'customer_name'])

joined = orders.join(customers, 'customer_id', 'left')
joined.show()

spark.stop()


## Window functions

Window functions are used for rankings, running totals, cumulative counts, and moving averages.


In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('WindowFunctionDemo').master('local[*]').getOrCreate()

df = spark.createDataFrame([
    ('A', '2024-01-01', 10),
    ('A', '2024-01-02', 15),
    ('A', '2024-01-03', 20),
    ('B', '2024-01-01', 50),
    ('B', '2024-01-02', 70)
], ['store', 'date', 'sales'])

window_spec = Window.partitionBy('store').orderBy('date')
df = df.withColumn('running_total', F.sum('sales').over(window_spec))
df.show()

spark.stop()


## Exercise
Create a small employee and department dataset. Compute total salary by department and rank employees by salary within each department.


In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('EmployeeExercise').master('local[*]').getOrCreate()

employees = spark.createDataFrame([
    (1, 'Alice', 100, 70000),
    (2, 'Bob', 100, 85000),
    (3, 'Charlie', 200, 90000),
    (4, 'Dana', 200, 60000)
], ['id', 'name', 'department_id', 'salary'])

dept_totals = employees.groupBy('department_id').agg(F.sum('salary').alias('dept_total'))
print('Department totals')
dept_totals.show()

window_spec = Window.partitionBy('department_id').orderBy(F.col('salary').desc())
ranked = employees.withColumn('salary_rank', F.row_number().over(window_spec))
ranked.show()

spark.stop()
